# Coleta bruta corrigida — RIPE Atlas (Períodos A e B)

Consulta somente GET, guarda respostas originais em JSON, separa os períodos A/B e preserva RTT ausente. Não calcula baseline nem classes. A execução precisa validar se as measurements configuradas correspondem ao mesh de Anchors e ao recorte temporal do guia.


## 1. Configuração e janela temporal

A janela segue o guia: 7 dias para A e os 7 dias seguintes para B. As datas abaixo reproduzem o recorte de referência informado no guia. Confirme se essas datas correspondem aos resultados disponíveis nas measurements escolhidas. A coleta não calcula baseline nem rótulos.


In [ ]:
from pathlib import Path

PASTA_SAIDA = Path('/content/drive/MyDrive/Coleta_RIPE_Atlas')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print("Pasta criada:", PASTA_SAIDA)
print("Existe?", PASTA_SAIDA.exists())

Pasta criada: /content/drive/MyDrive/Coleta_RIPE_Atlas
Existe? True


In [ ]:

import requests
import pandas as pd
import json
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

# Define a pasta na sua própria conta do Google Drive.
# O Drive já deve estar montado no Colab.
PASTA_PROJETO = Path('/content/drive/MyDrive/Coleta_RIPE_Atlas')
PASTA_RAW = PASTA_PROJETO / 'raw'

PASTA_RAW.mkdir(parents=True, exist_ok=True)

# Recorte do guia: 7 dias de A + 7 dias de B (UTC)
INICIO_A = datetime(2026, 9, 6, 4, 32, tzinfo=timezone.utc)
INICIO_B = INICIO_A + timedelta(days=7)
FIM_B = INICIO_B + timedelta(days=7)  # limite exclusivo

BLOCO_DIAS = 1
TIMEOUT_SEGUNDOS = 120
TENTATIVAS = 3
PAUSA_ENTRE_REQUISICOES = 0.5

# Mantenha aqui a lista FLUXOS original do seu notebook.

FLUXOS = [
    {
        "destino_pais": "Japão", "measurement_id": 109063401,
        "anchor_probe_id": 4220, "anchor_label": "JP",
        "probes_origem_pais": "FR",
        "probes_origem": [7825, 7796, 7819, 7202, 7286, 6333, 7048, 7146, 6320, 7604, 6717, 7507],
        "probe_referencia_informada": 7557,
    },
    {
        "destino_pais": "Brasil", "measurement_id": 25259597,
        "anchor_probe_id": 1690, "anchor_label": "BR",
        "probes_origem_pais": "BR",
        "probes_origem": [6977, 7113, 6410, 7708, 6602, 6659, 7508, 7019, 6891, 6349, 7307, 7242],
        "probe_referencia_informada": 6790,
    },
    {
        "destino_pais": "Canadá", "measurement_id": 23846774,
        "anchor_probe_id": 2057, "anchor_label": "CA",
        "probes_origem_pais": "US",
        "probes_origem": [6811, 7346, 6884, 7809, 7341, 7436, 6565, 6389, 7672, 7657, 6725, 6769],
        "probe_referencia_informada": 6719,
    },
    {
        "destino_pais": "Reino Unido", "measurement_id": 199622940,
        "anchor_probe_id": 4790, "anchor_label": "UK",
        "probes_origem_pais": "TR",
        "probes_origem": [7751, 7817, 7150, 7378, 6662, 6429, 6997, 7294, 7373, 7710, 7465, 7824],
        "probe_referencia_informada": 7789,
    },
    {
        "destino_pais": "Austrália", "measurement_id": 179906321,
        "anchor_probe_id": 4685, "anchor_label": "AU",
        "probes_origem_pais": "ID",
        "probes_origem": [6681, 7108, 6521, 7423, 7573, 7760, 7275, 7441, 7284, 7271, 7545, 7290],
        "probe_referencia_informada": 7767,
    },
]

assert len({f["measurement_id"] for f in FLUXOS}) == len(FLUXOS), "Há IDs de measurement repetidos."
for f in FLUXOS:
    assert len(f["probes_origem"]) == len(set(f["probes_origem"])), f"Há probes repetidas em {f['destino_pais']}."

print(f"Fluxos configurados: {len(FLUXOS)}")
print(f"Probes de origem selecionadas: {sum(len(f['probes_origem']) for f in FLUXOS)}")
print("Pasta do projeto:", PASTA_PROJETO.resolve())
print("Pasta dos dados brutos:", PASTA_RAW.resolve())


Fluxos configurados: 5
Probes de origem selecionadas: 60
Pasta do projeto: /content/drive/MyDrive/Coleta_RIPE_Atlas
Pasta dos dados brutos: /content/drive/MyDrive/Coleta_RIPE_Atlas/raw


## 2. Funções para consultar a API

O `prb_id` identifica a probe que originou o resultado. O ID da Anchor de destino informado pelo grupo é mantido como metadado para conferência, não como filtro de origem.


In [ ]:
BASE_API = "https://atlas.ripe.net/api/v2"

def requisitar_json(url, params=None):
    ultimo_erro = None
    for tentativa in range(TENTATIVAS):
        try:
            r = requests.get(url, params=params, timeout=TIMEOUT_SEGUNDOS)
            if r.status_code == 429 or 500 <= r.status_code < 600:
                r.raise_for_status()
            r.raise_for_status()
            return r.json()
        except Exception as e:
            ultimo_erro = e
            if tentativa + 1 < TENTATIVAS:
                time.sleep(2 ** tentativa)
    raise ultimo_erro

def obter_metadados(measurement_id):
    return requisitar_json(f"{BASE_API}/measurements/{measurement_id}/")

def extrair_resultados(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for k in ("results", "objects", "data"):
            if isinstance(payload.get(k), list):
                return payload[k]
    return []

def obter_resultados(measurement_id, inicio_epoch, fim_epoch, probe_ids):
    url = f"{BASE_API}/measurements/{measurement_id}/results/"
    params = {"start": int(inicio_epoch), "stop": int(fim_epoch), "probe_ids": ",".join(map(str, probe_ids)), "format": "json"}
    payload = requisitar_json(url, params=params)
    return payload, extrair_resultados(payload)

def slug(texto):
    return (str(texto).lower().replace(" ", "_").replace("á", "a")
            .replace("ã", "a").replace("é", "e").replace("í", "i")
            .replace("ó", "o").replace("ú", "u").replace("ç", "c"))


## 3. Conferir os metadados das measurements

Execute esta célula antes da coleta. Confira se `target_ip`, `type` e `description` combinam com o destino pretendido. Os metadados completos também serão salvos em JSON.


In [ ]:
resumo_meta = []
metadados_por_id = {}

for fluxo in FLUXOS:
    mid = fluxo["measurement_id"]
    try:
        meta = obter_metadados(mid)
        metadados_por_id[mid] = meta
        with open(PASTA_SAIDA / f"measurement_{mid}_metadados.json", "w", encoding="utf-8") as arq:
            json.dump(meta, arq, ensure_ascii=False, indent=2)
        resumo_meta.append({
            "destino_informado": fluxo["destino_pais"],
            "measurement_id": mid,
            "tipo_api": meta.get("type"),
            "af": meta.get("af"),
            "target_ip": meta.get("target_ip"),
            "validacao_ping_ipv4": (meta.get("type") == "ping" and meta.get("af") == 4),
            "target_ip": meta.get("target_ip"),
            "description": meta.get("description"),
            "status": meta.get("status"),
            "anchor_probe_id_informado": fluxo["anchor_probe_id"],
            "probes_origem_configuradas": len(fluxo["probes_origem"]),
            "consulta": "OK"
        })
    except Exception as e:
        resumo_meta.append({
            "destino_informado": fluxo["destino_pais"], "measurement_id": mid,
            "tipo_api": None, "target_ip": None, "description": None, "status": None,
            "anchor_probe_id_informado": fluxo["anchor_probe_id"],
            "probes_origem_configuradas": len(fluxo["probes_origem"]),
            "consulta": f"ERRO: {type(e).__name__}: {e}"
        })
    time.sleep(PAUSA_ENTRE_REQUISICOES)

df_metadados = pd.DataFrame(resumo_meta)
display(df_metadados)
df_metadados.to_csv(PASTA_SAIDA / "conferencia_metadados.csv", index=False, encoding="utf-8-sig")


,destino_informado,measurement_id,tipo_api,af,target_ip,validacao_ping_ipv4,description,status,anchor_probe_id_informado,probes_origem_configuradas,consulta
0,Japão,109063401,ping,4,103.96.67.60,True,Anchoring Mesh Measurement: Ping IPv4 for anch...,"{'id': 2, 'name': 'Ongoing', 'when': None}",4220,12,OK
1,Brasil,25259597,ping,4,45.11.105.136,True,Anchoring Mesh Measurement: Ping IPv4 for anch...,"{'id': 2, 'name': 'Ongoing', 'when': None}",1690,12,OK
2,Canadá,23846774,ping,4,54.39.185.31,True,Anchoring Mesh Measurement: Ping IPv4 for anch...,"{'id': 2, 'name': 'Ongoing', 'when': None}",2057,12,OK
3,Reino Unido,199622940,ping,4,23.26.247.12,True,Anchoring Mesh Measurement: Ping IPv4 for anch...,"{'id': 2, 'name': 'Ongoing', 'when': None}",4790,12,OK
4,Austrália,179906321,ping,4,103.52.117.1,True,Anchoring Mesh Measurement: Ping IPv4 for anch...,"{'id': 2, 'name': 'Ongoing', 'when': None}",4685,12,OK


## 4. Coleta por blocos diários, separando A e B

A API é consultada por dia para facilitar retomada. Os JSON de resposta são preservados. O filtro `probe_ids` é enviado à API e reaplicado localmente. A coleta não converte RTT ausente em zero e não atribui classes.


In [ ]:
def epoch(dt):
    return int(dt.timestamp())

def iter_blocos(inicio, fim, dias=1):
    atual = inicio
    while atual < fim:
        proximo = min(atual + timedelta(days=dias), fim)
        yield atual, proximo
        atual = proximo

relatorio = []
tabelas_filtradas = []
print("Período A UTC:", INICIO_A.isoformat(), "até", INICIO_B.isoformat())
print("Período B UTC:", INICIO_B.isoformat(), "até", FIM_B.isoformat())

for fluxo in FLUXOS:
    mid = fluxo["measurement_id"]
    prefixo = f"measurement_{mid}_{slug(fluxo['destino_pais'])}"
    probes = sorted(set(int(x) for x in fluxo["probes_origem"]))
    linha = {"destino_informado": fluxo["destino_pais"], "measurement_id": mid,
             "probes_escolhidas": len(probes), "resultados_brutos": 0,
             "resultados_filtrados": 0, "status": "", "erro": ""}
    try:
        meta = obter_metadados(mid)
        linha["tipo_api"] = meta.get("type")
        linha["af"] = meta.get("af")
        linha["target_ip_api"] = meta.get("target_ip")
        linha["validacao_ping_ipv4"] = (meta.get("type") == "ping" and meta.get("af") == 4)
        with open(PASTA_RAW / f"{prefixo}_metadados.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        if not linha["validacao_ping_ipv4"]:
            linha["aviso_metadados"] = "ATENÇÃO: confirmar type=ping e af=4 antes de usar no projeto"
        partes = []
        for nome_periodo, ini, fim in (("A", INICIO_A, INICIO_B), ("B", INICIO_B, FIM_B)):
            for bloco_ini, bloco_fim in iter_blocos(ini, fim, BLOCO_DIAS):
                payload, resultados = obter_resultados(mid, epoch(bloco_ini), epoch(bloco_fim), probes)
                sufixo = f"{nome_periodo}_{bloco_ini.strftime('%Y%m%d')}"
                with open(PASTA_RAW / f"{prefixo}_{sufixo}_bruto.json", "w", encoding="utf-8") as f:
                    json.dump(payload, f, ensure_ascii=False, indent=2)
                df = pd.json_normalize(resultados)
                if df.empty:
                    continue
                df["prb_id"] = pd.to_numeric(df.get("prb_id"), errors="coerce")
                df = df[df["prb_id"].isin(probes)].copy()
                df["measurement_id"] = mid
                df["destino_pais_informado"] = fluxo["destino_pais"]
                df["anchor_probe_id_informado"] = fluxo["anchor_probe_id"]
                df["pais_origem_esperado"] = fluxo["probes_origem_pais"]
                df["periodo"] = nome_periodo
                # Chave estável recomendada pelo guia; dst_addr vem da resposta da API.
                if "dst_addr" in df.columns:
                    df["fluxo_id"] = df["prb_id"].astype("Int64").astype(str) + "|" + df["dst_addr"].fillna("").astype(str)
                # Nunca preencher avg ausente com zero.
                if "avg" in df.columns:
                    df["latencia_ms"] = pd.to_numeric(df["avg"], errors="coerce")
                # UTC legível adicional, sem substituir timestamp epoch original.
                if "timestamp" in df.columns:
                    ts = pd.to_numeric(df["timestamp"], errors="coerce")
                    df["timestamp_utc"] = pd.to_datetime(ts, unit="s", utc=True, errors="coerce").astype(str)
                df.to_csv(PASTA_RAW / f"{prefixo}_{sufixo}_probes_selecionadas.csv", index=False, encoding="utf-8-sig")
                partes.append(df)
                linha["resultados_brutos"] += len(resultados)
                linha["resultados_filtrados"] += len(df)
                time.sleep(PAUSA_ENTRE_REQUISICOES)
        if partes:
            df_fluxo = pd.concat(partes, ignore_index=True, sort=False)
            df_fluxo.to_csv(PASTA_RAW / f"{prefixo}_A_B_probes_selecionadas.csv", index=False, encoding="utf-8-sig")
            tabelas_filtradas.append(df_fluxo)
            linha["probes_com_resultado"] = int(df_fluxo["prb_id"].nunique()) if "prb_id" in df_fluxo else 0
            linha["status"] = "OK" if len(df_fluxo) else "SEM RESULTADOS"
        else:
            linha["status"] = "SEM RESULTADOS NO INTERVALO CONFIGURADO"
    except Exception as e:
        linha["status"] = "ERRO"
        linha["erro"] = f"{type(e).__name__}: {e}"
    relatorio.append(linha)

if tabelas_filtradas:
    df_consolidado = pd.concat(tabelas_filtradas, ignore_index=True, sort=False)
else:
    df_consolidado = pd.DataFrame()
df_consolidado.to_csv(PASTA_RAW / "todos_fluxos_probes_selecionadas_A_B.csv", index=False, encoding="utf-8-sig")
df_relatorio = pd.DataFrame(relatorio)
display(df_relatorio)
df_relatorio.to_csv(PASTA_RAW / "relatorio_coleta.csv", index=False, encoding="utf-8-sig")
df_relatorio.to_json(PASTA_RAW / "relatorio_coleta.json", orient="records", force_ascii=False, indent=2)
print("Arquivos salvos em:", PASTA_RAW.resolve())


Período A UTC: 2026-09-06T04:32:00+00:00 até 2026-09-13T04:32:00+00:00
Período B UTC: 2026-09-13T04:32:00+00:00 até 2026-09-20T04:32:00+00:00


,destino_informado,measurement_id,probes_escolhidas,resultados_brutos,resultados_filtrados,status,erro,tipo_api,af,target_ip_api,validacao_ping_ipv4,probes_com_resultado
0,Japão,109063401,12,50313,50313,OK,,ping,4,103.96.67.60,True,10
1,Brasil,25259597,12,60349,60349,OK,,ping,4,45.11.105.136,True,12
2,Canadá,23846774,12,55360,55360,OK,,ping,4,54.39.185.31,True,11
3,Reino Unido,199622940,12,50274,50274,OK,,ping,4,23.26.247.12,True,10
4,Austrália,179906321,12,59028,59028,OK,,ping,4,103.52.117.1,True,12


Arquivos salvos em: /content/drive/MyDrive/Coleta_RIPE_Atlas/raw


## 5. Consolidação e auditoria inicial

O CSV consolidado contém registros dos períodos A e B, com a coluna `periodo`. Ele não calcula baseline, métricas relativas nem rótulos. Verifique a validação de metadados e a disponibilidade real de resultados antes de usar os dados.


In [ ]:
if not df_consolidado.empty:
    display(df_consolidado.head(10))
    print("Total de linhas:", len(df_consolidado))
    print("Por período:", df_consolidado["periodo"].value_counts(dropna=False).to_dict())
    if "latencia_ms" in df_consolidado:
        print("RTT ausentes (mantidos como ausentes):", int(df_consolidado["latencia_ms"].isna().sum()))
else:
    print("Nenhum resultado encontrado. Confira o relatório, as datas e os metadados das measurements.")


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,step,stored_timestamp,measurement_id,destino_pais_informado,anchor_probe_id_informado,pais_origem_esperado,periodo,fluxo_id,latencia_ms,timestamp_utc
0,5130,2.6.4,13,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788739280,109063401,Japão,4220,FR,A,6320|103.96.67.60,249.705483,2026-09-07 00:00:30+00:00
1,5130,2.6.4,4,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788739528,109063401,Japão,4220,FR,A,6320|103.96.67.60,248.641990,2026-09-07 00:04:25+00:00
2,5130,2.6.4,68,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788739768,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.768090,2026-09-07 00:08:25+00:00
3,5130,2.6.4,67,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788739995,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.796851,2026-09-07 00:12:19+00:00
4,5130,2.6.4,66,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788740233,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.203266,2026-09-07 00:16:27+00:00
5,5130,2.6.4,10,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788740480,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.789711,2026-09-07 00:20:25+00:00
6,5130,2.6.4,12,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788740729,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.238582,2026-09-07 00:24:23+00:00
7,5130,2.6.4,10,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788740970,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.715884,2026-09-07 00:28:28+00:00
8,5130,2.6.4,4,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788741192,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.367795,2026-09-07 00:32:25+00:00
9,5130,2.6.4,1,103.96.67.60,4,103.96.67.60,185.233.100.99,ICMP,45.0,64,...,240,1788741445,109063401,Japão,4220,FR,A,6320|103.96.67.60,244.083243,2026-09-07 00:36:19+00:00


Total de linhas: 275324
Por período: {'B': 138049, 'A': 137275}
RTT ausentes (mantidos como ausentes): 0


## 6. Checklist antes de usar os dados

- [ ] Confirmar em `relatorio_coleta.csv` que cada measurement é `ping` e `af=4`.
- [ ] Confirmar que os destinos reais (`target_ip`/`dst_addr`) correspondem ao desenho de fluxos do grupo.
- [ ] Conferir se há resultados nos 14 dias configurados; uma measurement pode não ter histórico nesse intervalo.
- [ ] Verificar que A e B não se sobrepõem e que cada linha tem a coluna `periodo`.
- [ ] RTT ausente continua ausente; não substituir por zero.
- [ ] Registros com `sent` presente e perda de 100% devem permanecer na coleta.
- [ ] Na Tarefa 2, calcular baseline por fluxo usando apenas A e exigir 1.500 RTTs válidos; não calcular classes nesta coleta.
- [ ] Preservar os JSON brutos e não editar os arquivos originais.

**Limitação importante:** esta correção configura o recorte de datas do guia, mas não prova que os IDs de measurement selecionados correspondem ao mesh de Anchors descrito. Se os metadados falharem ou não houver resultados nessas datas, não use os CSVs como se estivessem validados; será necessário corrigir a seleção de measurements/fluxos.
